# WAFER-GUARD · Fab-7 Excursion Detector — Repaired Pipeline

This notebook replaces the original release candidate. It fixes three silent defects:

| # | Defect | Where (original) | Fix here |
|---|---|---|---|
| 1 | **Data leakage**: imputer/scaler fit on all rows, SMOTE run before the split, random (not time-ordered) split | §2 | Time-ordered train/val/test split; all fitting uses train rows only; no synthetic samples; test set stays at natural prevalence |
| 2 | **Corrupted loss gradients**: `reconstructed` is `[B, F]` but `batch_x` is `[B, 1, F]`, so the MSE broadcasts to `[B, B, F]` | §5 | Inputs kept as `[B, F]`; masked per-element MSE with explicit shape asserts |
| 3 | **Invariant blindness**: legacy probe station PS-L writes `sensor_0 = 0.0` (sentinel) and puts firmware codes 100/200/300 in `sensor_1`; the original treats both as real measurements | §2 | Detect legacy rows, one-hot encode the codes, mask both channels out of scaling, imputation and reconstruction loss |

Section 7 reproduces each defect with numbers from this run.

## 0 · Environment

In [1]:
# !pip install "torch>=2.2" "scikit-learn>=1.2" "imbalanced-learn>=0.11" pandas numpy joblib
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             average_precision_score, confusion_matrix, precision_recall_curve)
import joblib

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 1337
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
seed_everything()
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cpu")   # CPU for bit-for-bit reproducibility

# hyper-parameters picked by a small sweep scored on VALIDATION PR-AUC only (see AUDIT.md)
HP = dict(batch_size=64, epochs=30, lr=1e-4, weight_decay=1e-2,
          hidden=256, latent=8, lambda_clf=1.0, dropout=0.3,
          train_frac=0.60, val_frac=0.20)          # test = last 20 % in time
print(f"torch {torch.__version__} | device={DEVICE}")

torch 2.14.1+cu130 | device=cpu


## 1 · Data — loaded in time order

Rows are sorted by `timestamp` so every split respects causality: the model is
always trained on the past and scored on the future.

In [2]:
df = pd.read_csv("secom_telemetry_modified.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)

SENSOR_COLS = [c for c in df.columns if c.startswith("sensor_")]
X_all = df[SENSOR_COLS].to_numpy(dtype=np.float64)             # [N, 590]
y_all = (df["label"].to_numpy() == 1).astype(np.int64)          # 1 = defective
N = len(df)
print(f"X: {X_all.shape} | defect rate: {y_all.mean():.2%} ({y_all.sum()} fail / {N - y_all.sum()} pass)")
print(f"time range: {df.timestamp.min()} -> {df.timestamp.max()}")

X: (1567, 590) | defect rate: 6.64% (104 fail / 1463 pass)
time range: 2008-07-19 11:55:00 -> 2008-10-17 06:07:00


## 2 · Legacy probe station PS-L (tool-matching invariant)

PS-L firmware writes `sensor_0 == 0.0` exactly. That is a sentinel, not a measurement.
On those wafers `sensor_1` holds a firmware error code (100 / 200 / 300), not a
continuous reading. The check below is a fixed, per-row rule, so it uses no
information from other rows or from the future.

Handling:
* `is_legacy` flag and a one-hot of the code become **categorical** inputs.
* For legacy rows, `sensor_0` and `sensor_1` are set to *missing*. They are left
  out of scaler statistics and imputation, and their reconstruction error is
  masked out of the loss. The model never sees a made-up continuous value for them.

In [3]:
LEGACY_CODES = [100.0, 200.0, 300.0]
is_legacy = (X_all[:, 0] == 0.0)
codes = X_all[is_legacy, 1]
print(f"legacy wafers: {is_legacy.sum()} ({is_legacy.mean():.2%})")
print("sensor_1 values on legacy rows:", dict(zip(*np.unique(codes, return_counts=True))))
assert np.isin(codes, LEGACY_CODES).all(), "unexpected PS-L firmware code"
print(f"non-legacy sensor_1 range: {np.nanmin(X_all[~is_legacy,1]):.1f} .. {np.nanmax(X_all[~is_legacy,1]):.1f}")
print(f"defect rate  legacy={y_all[is_legacy].mean():.2%}  current-gen={y_all[~is_legacy].mean():.2%}")

# categorical block: [is_legacy, code_100, code_200, code_300]
CAT = np.zeros((N, 1 + len(LEGACY_CODES)), dtype=np.float64)
CAT[:, 0] = is_legacy
for j, c in enumerate(LEGACY_CODES):
    CAT[:, 1 + j] = is_legacy & (X_all[:, 1] == c)
CAT_NAMES = ["is_legacy"] + [f"ps_l_code_{int(c)}" for c in LEGACY_CODES]

# continuous block: the sentinel and the code channel are *not measurements* on legacy rows
X_cont = X_all.copy()
X_cont[is_legacy, 0] = np.nan
X_cont[is_legacy, 1] = np.nan

legacy wafers: 146 (9.32%)
sensor_1 values on legacy rows: {np.float64(100.0): np.int64(61), np.float64(200.0): np.int64(35), np.float64(300.0): np.int64(50)}
non-legacy sensor_1 range: 2158.8 .. 2846.4
defect rate  legacy=0.00%  current-gen=7.32%


## 3 · Time-ordered split, then preprocessing fit on train only

* Train = first 60 % of lots, validation = next 20 %, test = last 20 %. No shuffling across time.
* Column filtering, median imputation and standardisation are fit on **train rows only**,
  then applied unchanged to val and test, the same way they would run in production.
* There is no SMOTE: class imbalance is handled with `pos_weight` in the loss. Val and
  test stay at the natural defect rate.

In [4]:
n_tr = int(N * HP["train_frac"]); n_va = int(N * (HP["train_frac"] + HP["val_frac"]))
idx_tr, idx_va, idx_te = np.arange(0, n_tr), np.arange(n_tr, n_va), np.arange(n_va, N)
assert df.timestamp.iloc[idx_tr].max() <= df.timestamp.iloc[idx_va].min() <= df.timestamp.iloc[idx_te].min()
for name, idx in [("train", idx_tr), ("val", idx_va), ("test", idx_te)]:
    print(f"{name:5s}: n={len(idx):4d} | defects={y_all[idx].sum():3d} ({y_all[idx].mean():.2%}) "
          f"| legacy={is_legacy[idx].sum():3d} | {df.timestamp.iloc[idx[0]]} -> {df.timestamp.iloc[idx[-1]]}")


class CausalPreprocessor:
    """Fit on the training window only. transform() works on one row at a time (no cross-row state)."""
    def fit(self, Xc):
        obs = ~np.isnan(Xc)
        frac_obs = obs.mean(0)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            std = np.nanstd(Xc, axis=0)
        # drop channels that are mostly empty or constant *in the training window*
        self.keep = (frac_obs >= 0.5) & (np.nan_to_num(std, nan=0.0) > 1e-8)
        Xk = Xc[:, self.keep]
        self.median = np.nanmedian(Xk, axis=0)
        self.mean = np.nanmean(Xk, axis=0)
        self.std = np.nanstd(Xk, axis=0)
        return self

    def transform(self, Xc):
        Xk = Xc[:, self.keep]
        mask = ~np.isnan(Xk)                                  # 1 = genuinely observed
        Xk = np.where(mask, Xk, self.median)                  # train-median imputation
        Z = (Xk - self.mean) / self.std
        Z = np.clip(Z, -10, 10)                               # bound out-of-range drift
        Z[~mask] = 0.0                                        # unobserved -> neutral, flagged via mask
        return Z.astype(np.float32), mask.astype(np.float32)


prep = CausalPreprocessor().fit(X_cont[idx_tr])
kept_names = np.array(SENSOR_COLS)[prep.keep]
print(f"continuous channels kept: {prep.keep.sum()} / {len(SENSOR_COLS)} "
      f"(sensor_0 kept={prep.keep[0]}, sensor_1 kept={prep.keep[1]})")

def build(idx):
    Z, M = prep.transform(X_cont[idx])
    return Z, M, CAT[idx].astype(np.float32), y_all[idx].astype(np.float32)

Z_tr, M_tr, C_tr, y_tr = build(idx_tr)
Z_va, M_va, C_va, y_va = build(idx_va)
Z_te, M_te, C_te, y_te = build(idx_te)

# PS-L channels are masked, never imputed into the loss
leg_tr = is_legacy[idx_tr]
assert (M_tr[leg_tr][:, :2] == 0).all() if prep.keep[:2].all() else True
print(f"masked (non-observed) cells in train: {(1 - M_tr).mean():.2%}")

train: n= 940 | defects= 76 (8.09%) | legacy= 79 | 2008-07-19 11:55:00 -> 2008-09-20 05:34:00
val  : n= 313 | defects= 11 (3.51%) | legacy= 40 | 2008-09-20 06:08:00 -> 2008-10-02 19:25:00
test : n= 314 | defects= 17 (5.41%) | legacy= 27 | 2008-10-02 20:54:00 -> 2008-10-17 06:07:00
continuous channels kept: 444 / 590 (sensor_0 kept=True, sensor_1 kept=True)
masked (non-observed) cells in train: 2.05%


## 4 · Tensors & loaders — flat `[B, F]` everywhere

In [5]:
F_CONT, F_CAT = Z_tr.shape[1], C_tr.shape[1]

def make_loader(Z, M, C, y, shuffle):
    tensors = [torch.from_numpy(a) for a in (Z, M, C, y)]
    assert tensors[0].ndim == 2 and tensors[3].ndim == 1      # [N, F] and [N]
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(TensorDataset(*tensors), batch_size=HP["batch_size"], shuffle=shuffle,
                      generator=g if shuffle else None, drop_last=False)

train_loader = make_loader(Z_tr, M_tr, C_tr, y_tr, shuffle=True)
val_loader   = make_loader(Z_va, M_va, C_va, y_va, shuffle=False)
test_loader  = make_loader(Z_te, M_te, C_te, y_te, shuffle=False)
print(f"F_cont={F_CONT} | F_cat={F_CAT} | train batches={len(train_loader)}")

F_cont=444 | F_cat=4 | train batches=15


## 5 · Model — supervised autoencoder

The encoder sees `[continuous ‖ categorical]`. The decoder rebuilds only the **continuous**
channels. The reconstruction loss is a masked per-element MSE: imputed cells and PS-L
sentinel/code cells add zero error and zero gradient.

In [6]:
class WaferSAE(nn.Module):
    def __init__(self, d_cont, d_cat, d_hid, d_lat, p_drop):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_cont + d_cat, d_hid), nn.LayerNorm(d_hid), nn.GELU(), nn.Dropout(p_drop),
            nn.Linear(d_hid, d_hid // 2), nn.LayerNorm(d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_lat),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_lat, d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_hid), nn.GELU(),
            nn.Linear(d_hid, d_cont),
        )
        self.head = nn.Linear(d_lat, 1)

    def forward(self, x_cont, x_cat):
        assert x_cont.ndim == 2 and x_cat.ndim == 2, f"expected [B,F], got {tuple(x_cont.shape)}"
        z = self.encoder(torch.cat([x_cont, x_cat], dim=1))
        return self.decoder(z), self.head(z).squeeze(-1), z      # [B,F_cont], [B], [B,lat]


def masked_mse(recon, target, mask):
    assert recon.shape == target.shape == mask.shape, (recon.shape, target.shape, mask.shape)
    se = (recon - target) ** 2 * mask
    return se.sum() / mask.sum().clamp_min(1.0)

seed_everything()
model = WaferSAE(F_CONT, F_CAT, HP["hidden"], HP["latent"], HP["dropout"]).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

parameters: 297,933


## 6 · Training — model selection on validation PR-AUC

Class weight `pos_weight = n_neg / n_pos` comes from the training window only.
The checkpoint with the best **validation** average precision is kept. The test window
is not touched until §8.

In [7]:
pos_weight = torch.tensor((1 - y_tr).sum() / y_tr.sum(), dtype=torch.float32)
bce  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt  = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HP["epochs"])
print(f"pos_weight = {pos_weight.item():.2f}")

@torch.no_grad()
def predict(loader):
    model.eval(); out = []
    for zb, mb, cb, _ in loader:
        _, logits, _ = model(zb.to(DEVICE), cb.to(DEVICE))
        out.append(torch.sigmoid(logits).cpu())
    return torch.cat(out).numpy()

history = {"total": [], "recon": [], "clf": [], "val_ap": []}
best = dict(ap=-1.0, epoch=0, state=None)

for epoch in range(1, HP["epochs"] + 1):
    model.train(); acc = dict(total=0.0, recon=0.0, clf=0.0); nb = 0
    for zb, mb, cb, yb in train_loader:
        zb, mb, cb, yb = zb.to(DEVICE), mb.to(DEVICE), cb.to(DEVICE), yb.to(DEVICE)
        recon, logits, _ = model(zb, cb)
        assert logits.shape == yb.shape, (logits.shape, yb.shape)
        recon_loss = masked_mse(recon, zb, mb)
        clf_loss   = bce(logits, yb)
        loss       = recon_loss + HP["lambda_clf"] * clf_loss
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        acc["total"] += loss.item(); acc["recon"] += recon_loss.item(); acc["clf"] += clf_loss.item(); nb += 1
    sched.step()
    for k in acc: history[k].append(acc[k] / nb)
    val_ap = average_precision_score(y_va, predict(val_loader)); history["val_ap"].append(val_ap)
    if val_ap > best["ap"]:
        best = dict(ap=val_ap, epoch=epoch, state={k: v.clone() for k, v in model.state_dict().items()})
    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:02d} | total {history['total'][-1]:.4f} | recon {history['recon'][-1]:.4f} "
              f"| clf {history['clf'][-1]:.4f} | val AP {val_ap:.4f}")

model.load_state_dict(best["state"])
print(f"best epoch {best['epoch']} | val AP {best['ap']:.4f}")

pos_weight = 11.37


epoch 01 | total 2.1184 | recon 0.8586 | clf 1.2598 | val AP 0.0767


epoch 05 | total 1.8556 | recon 0.8562 | clf 0.9994 | val AP 0.0600


epoch 10 | total 1.5767 | recon 0.8460 | clf 0.7307 | val AP 0.0486


epoch 15 | total 1.3819 | recon 0.8417 | clf 0.5402 | val AP 0.0451


epoch 20 | total 1.2694 | recon 0.8374 | clf 0.4320 | val AP 0.0450


epoch 25 | total 1.2303 | recon 0.8353 | clf 0.3950 | val AP 0.0447


epoch 30 | total 1.2177 | recon 0.8374 | clf 0.3802 | val AP 0.0445
best epoch 2 | val AP 0.0850


## 7 · Threshold — chosen on validation only

The decision threshold is the one that gives the best F1 on the **validation** window.
It is then frozen and applied to test. Choosing it on test would leak test labels.

In [8]:
p_va = predict(val_loader)
prec_c, rec_c, thr_c = precision_recall_curve(y_va, p_va)
f1_c = 2 * prec_c[:-1] * rec_c[:-1] / np.clip(prec_c[:-1] + rec_c[:-1], 1e-12, None)
THRESH = float(thr_c[np.nanargmax(f1_c)])
print(f"frozen threshold = {THRESH:.4f} | val F1 at threshold = {np.nanmax(f1_c):.4f}")

frozen threshold = 0.6278 | val F1 at threshold = 0.1429


## 8 · Final evaluation — held-out future window at natural prevalence

Main metric: **PR-AUC (average precision)**, compared with the no-skill baseline (= prevalence).
At a ~6.6 % defect rate, accuracy and ROC-AUC look good even for weak models, so they are not the main measure.
F1, precision, recall and the confusion matrix are reported at the frozen threshold.

In [9]:
p_te = predict(test_loader)
pred_te = (p_te >= THRESH).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te, pred_te, labels=[0, 1]).ravel()
RESULTS = dict(
    test_n=len(y_te), test_defects=int(y_te.sum()), test_prevalence=float(y_te.mean()),
    threshold=THRESH,
    f1=f1_score(y_te, pred_te, zero_division=0),
    precision=precision_score(y_te, pred_te, zero_division=0),
    recall=recall_score(y_te, pred_te, zero_division=0),
    roc_auc=roc_auc_score(y_te, p_te),
    pr_auc=average_precision_score(y_te, p_te),
    TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp),
)
for k, v in RESULTS.items():
    print(f"{k:16s} {v:.4f}" if isinstance(v, float) else f"{k:16s} {v}")
print(f"\nPR-AUC lift over no-skill baseline: {RESULTS['pr_auc'] / RESULTS['test_prevalence']:.2f}x")

test_n           314
test_defects     17
test_prevalence  0.0541
threshold        0.6278
f1               0.0000
precision        0.0000
recall           0.0000
roc_auc          0.4977
pr_auc           0.0589
TN               293
FP               4
FN               17
TP               0

PR-AUC lift over no-skill baseline: 1.09x


## 9 · Evidence: reproducing the original defects

Every number below is computed in this run.

In [10]:
# --- Defect 2: broadcast in the original reconstruction loss ---
xb = torch.randn(HP["batch_size"], 1, 590)          # original loader: unsqueeze(1) -> [B,1,F]
rec = torch.randn(HP["batch_size"], 590)            # original model: flatten -> [B,F]
diff = rec - xb
print(f"[bug 2] (reconstructed - batch_x).shape = {tuple(diff.shape)}  "
      f"<- should be ({HP['batch_size']}, 590); every sample is compared to every other sample")
rec.requires_grad_(True)
g_bad,  = torch.autograd.grad(torch.mean((rec - xb) ** 2), rec)
g_good, = torch.autograd.grad(torch.mean((rec - xb.squeeze(1)) ** 2), rec)
cos = torch.nn.functional.cosine_similarity(g_bad.flatten(), g_good.flatten(), dim=0)
# the broken loss is minimised by reconstructing the *batch mean* for every sample
g_mean, = torch.autograd.grad(torch.mean((rec - xb.mean(0)) ** 2), rec)
cos_mean = torch.nn.functional.cosine_similarity(g_bad.flatten(), g_mean.flatten(), dim=0)
print(f"[bug 2] cosine(grad_broken, grad_correct) = {cos.item():.3f}; "
      f"cosine(grad_broken, grad_toward_batch_mean) = {cos_mean.item():.3f}")
print("[bug 2] -> the broken loss trains the decoder to output the batch average, not to reconstruct each wafer")

[bug 2] (reconstructed - batch_x).shape = (64, 64, 590)  <- should be (64, 590); every sample is compared to every other sample
[bug 2] cosine(grad_broken, grad_correct) = 0.714; cosine(grad_broken, grad_toward_batch_mean) = 1.000
[bug 2] -> the broken loss trains the decoder to output the batch average, not to reconstruct each wafer


In [11]:
# --- Defect 3: PS-L sentinel / codes treated as measurements ---
raw0, raw1 = X_all[:, 0], X_all[:, 1]
print(f"[bug 3] sensor_0  mean/std  with sentinel: {np.nanmean(raw0):8.1f} / {np.nanstd(raw0):7.1f}"
      f"   | current-gen only: {np.nanmean(raw0[~is_legacy]):8.1f} / {np.nanstd(raw0[~is_legacy]):7.1f}")
print(f"[bug 3] sensor_1  mean/std  with codes   : {np.nanmean(raw1):8.1f} / {np.nanstd(raw1):7.1f}"
      f"   | current-gen only: {np.nanmean(raw1[~is_legacy]):8.1f} / {np.nanstd(raw1[~is_legacy]):7.1f}")
z_leg = (raw0[is_legacy][0] - np.nanmean(raw0)) / np.nanstd(raw0)
print(f"[bug 3] original scaling puts a PS-L sentinel at z = {z_leg:.2f}: an extreme fake excursion, "
      f"and code 300 is read as 'more' than code 100")

[bug 3] sensor_0  mean/std  with sentinel:   2732.1 /   880.4   | current-gen only:   3014.0 /    74.3
[bug 3] sensor_1  mean/std  with codes   :   2280.5 /   675.1   | current-gen only:   2495.7 /    79.8
[bug 3] original scaling puts a PS-L sentinel at z = -3.10: an extreme fake excursion, and code 300 is read as 'more' than code 100


In [12]:
# --- Defect 1: leakage. Re-run the ORIGINAL preprocessing order and measure what it does ---
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

X_raw = pd.read_csv("secom_telemetry_modified.csv")[SENSOR_COLS].to_numpy(np.float64)
y_raw = (pd.read_csv("secom_telemetry_modified.csv")["label"].to_numpy() == 1).astype(int)
Xs = StandardScaler().fit_transform(SimpleImputer(strategy="mean", keep_empty_features=True).fit_transform(X_raw))
Xb, yb_ = SMOTE(random_state=SEED).fit_resample(Xs, y_raw)
Xtr_o, Xte_o, ytr_o, yte_o = train_test_split(Xb, yb_, test_size=0.2, stratify=yb_, random_state=SEED)
n_real = len(X_raw)
synthetic_in_test = np.sum([~np.any(np.all(np.isclose(Xs, r), axis=1)) for r in Xte_o[yte_o == 1]])
print(f"[bug 1] original test set: n={len(yte_o)} | defect rate {yte_o.mean():.1%} (natural is {y_raw.mean():.1%})")
print(f"[bug 1] {synthetic_in_test}/{(yte_o==1).sum()} test 'defects' are SMOTE synthetics "
      f"built by interpolating between training defects")
print(f"[bug 1] imputer/scaler were fit on all {n_real} rows, including future and test lots")
print(f"[bug 1] the random split mixes lots from {df.timestamp.min().date()} to {df.timestamp.max().date()} into train and test")

[bug 1] original test set: n=586 | defect rate 50.0% (natural is 6.6%)
[bug 1] 274/293 test 'defects' are SMOTE synthetics built by interpolating between training defects
[bug 1] imputer/scaler were fit on all 1567 rows, including future and test lots
[bug 1] the random split mixes lots from 2008-07-19 to 2008-10-17 into train and test


In [13]:
# --- Ablation: the same SAE recipe scored the original way vs the honest way ---
from sklearn.linear_model import LogisticRegression
lr_leaky = LogisticRegression(max_iter=2000, C=0.1).fit(Xtr_o, ytr_o)
f1_leaky = f1_score(yte_o, lr_leaky.predict(Xte_o))
lr_honest = LogisticRegression(max_iter=2000, C=0.1, class_weight="balanced").fit(
    np.hstack([Z_tr, C_tr]), y_tr)
p_lr = lr_honest.predict_proba(np.hstack([Z_te, C_te]))[:, 1]
print(f"[ablation] logistic baseline, original protocol (SMOTE→split, balanced test): F1 = {f1_leaky:.3f}")
print(f"[ablation] logistic baseline, causal protocol (future test, 6.6 % prevalence): "
      f"PR-AUC = {average_precision_score(y_te, p_lr):.3f}, ROC-AUC = {roc_auc_score(y_te, p_lr):.3f}")
print("→ Even a linear model 'passes' the F1 ≥ 0.90 gate under the original protocol: the gate measured the leakage, not the model.")

[ablation] logistic baseline, original protocol (SMOTE→split, balanced test): F1 = 0.932
[ablation] logistic baseline, causal protocol (future test, 6.6 % prevalence): PR-AUC = 0.090, ROC-AUC = 0.642
→ Even a linear model 'passes' the F1 ≥ 0.90 gate under the original protocol: the gate measured the leakage, not the model.


## 10 · Release gate & artifacts

In [14]:
GATE_F1 = 0.90
verdict = "PASSED ✅" if RESULTS["f1"] >= GATE_F1 else "FAILED ❌ — NO-GO"
print(f"Release gate (F1 ≥ {GATE_F1}) on honest future test window: {RESULTS['f1']:.4f} → {verdict}")

os.makedirs("artifacts", exist_ok=True)
torch.save(model.state_dict(), "artifacts/wafer_sae_fixed.pt")
joblib.dump({"preprocessor": prep, "legacy_codes": LEGACY_CODES, "threshold": THRESH,
             "cat_names": CAT_NAMES, "kept_sensors": kept_names.tolist()}, "artifacts/preprocessor_fixed.joblib")
print("Artifacts written to ./artifacts/")

Release gate (F1 ≥ 0.9) on honest future test window: 0.0000 → FAILED ❌ — NO-GO
Artifacts written to ./artifacts/
